# ORP AI — Fine-tune Deteksi Tuberkulosis (Colab, CPU/GPU)
**Proyek:** ORP RIS/PACS (`ai-worker`) — pembuatan alat skrining TB (bukan alat diagnostik).
**Data:** Montgomery County (138 CXR) + Shenzhen (662 CXR) — dataset **publik** NLM/OpenI.

> **Aturan penggunaan penting**
> 1. Notebook ini HANYA untuk dataset publik (Montgomery/Shenzhen/TBX11K).
> 2. **JANGAN pernah upload data pasien / institusi ke Colab** (UU PDP 27/2022, kebijakan privasi workspace).
> 3. Data hidup di runtime Colab (ephemeral). Yang keluar dari Colab HANYA **bobot model** (±35 MB) via Google Drive / unduhan.
> 4. Hasil akhir = alat **skrining/triase**, bukan diagnosis. Perlu validasi klinis sebelum dipakai produksi.
>
> Referensi fakta terverifikasi: file rencana proyek `Rencana Pembangunan — ORP RIS.md` §6a.

In [ ]:
# @markdown ## 1. Cek environment
import platform, sys
import torch
print("Python :", platform.python_version())
print("torch  :", torch.__version__)
print("Device :", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("RAM    :", round(__import__('psutil').virtual_memory().total/1e9, 1), "GB (perkiraan)")


Python : 3.13.15
torch  : 2.11.0+cpu
Device : CPU
RAM    : 13.6 GB (perkiraan)


In [ ]:
# @markdown ## 2. Instal dependensi & unduh dataset → ke /content (BUKAN penyimpanan lokal Anda)
%pip -q install huggingface_hub scikit-learn matplotlib
from huggingface_hub import snapshot_download

# Dataset publik Montgomery+Shenzhen, label di filename/ClinicalReadings (paket HF).
# Ukuran ±100 MB. Diunduh langsung ke disk runtime Colab.
import os
os.makedirs("/content/data", exist_ok=True)
repo = snapshot_download(
    repo_id="Famatsu123/montgomery-shenzhen-tuberculosis-cxr",
    repo_type="dataset",
    local_dir="/content/data/monty_shenzhen",
)
print("Dataset siap di:", repo)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1887 files:   0%|          | 0/1887 [00:00<?, ?it/s]

Dataset siap di: /content/data/monty_shenzhen


In [ ]:
# @markdown ## 3. Inspeksi struktur & skema label (SEBELUM training — selalu cek dulu)
import os
from pathlib import Path

root = Path("/content/data/monty_shenzhen")
imgs = list(root.rglob("*.[pP][nN][gG]")) + list(root.rglob("*.[jJ][pP][gG]"))
print("Total gambar:", len(imgs))
print("\n--- contoh nama file ---")
for p in imgs[:10]:
    print(p.relative_to(root))
print("\n--- teks ClinicalReadings (Montgomery) ---")
for txt in root.rglob("*.txt"):
    print(f"[{txt.name}]:", txt.read_text().splitlines()[:4])
    if txt.name.startswith("MCUCXR"):  # cukup contoh
        break


In [ ]:
# @markdown ## 4. Pemetaan label (heuristik multi-pola + verifikasi distribusi)
import pandas as pd

def label_from_path(p: Path) -> str | None:
    name = p.name.lower()
    if "tb" in name and "normal" not in name:
        return "tb"
    if "normal" in name and "tb" not in name:
        return "normal"
    if "tuberculosis" in name:
        return "tb"
    # Montgomery: ClinicalReadings/*.txt berisi Sex/Age/Diagnosis
    txt = p.parent.parent / "ClinicalReadings" / (p.stem.split("_")[0] + ".txt")
    for cand in [txt, p.parent / "ClinicalReadings" / (p.stem + ".txt")]:
        pass
    return None

rows = []
for p in imgs:
    label = label_from_path(p)
    if label is None:
        # fallback: baca txt Montgomery dengan nama sama di folder ClinicalReadings
        cr = root / "MontgomerySet" / "ClinicalReadings"
        if cr.exists():
            for txt in cr.glob(p.stem + ".txt"):
                lines = txt.read_text().splitlines()
                if len(lines) >= 3:
                    diag = lines[2].strip().lower()
                    label = "tb" if diag in ("tb", "tuberculosis", "tb+") else "normal"
    rows.append((str(p), label))

df = pd.DataFrame(rows, columns=["path", "label"])
print(df["label"].value_counts(dropna=False))
print("\nContoh yang tidak ter-label (cek manual jika banyak):")
print(df[df["label"].isna()].head(5).to_string())
df = df.dropna(subset=["label"]).reset_index(drop=True)
df.to_csv("/content/data/labels.csv", index=False)
print("\nDipakai:", len(df), "gambar")


In [ ]:
# @markdown ## 5. Dataset PyTorch + split stratify (seed tetap → reproducible)
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # CXR → 3-channel untuk DenseNet ImageNet
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class CXR(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True)
        self.tf = tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        img = Image.open(self.df.loc[i, "path"]).convert("L")
        y = 1.0 if self.df.loc[i, "label"] == "tb" else 0.0
        return self.tf(img), torch.tensor(y, dtype=torch.float32)

tr, va = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
train_ds, val_ds = CXR(tr, tf), CXR(va, tf)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_dl   = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)
print(f"train={len(tr)} ({tr['label'].value_counts().to_dict()})  val={len(va)} ({va['label'].value_counts().to_dict()})")


In [ ]:
# @markdown ## 6. Fine-tune DenseNet121 (ImageNet) → 1 output BCE
# @markdown 20 epoch CPU Colab ±10-20 menit; GPU T4 ±2 menit.
from torchvision import models
from torch import nn

def build_model():
    m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    m.classifier = nn.Linear(m.classifier.in_features, 1)  # BCEWithLogits → output logit tunggal
    return m

model = build_model()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
crit = nn.BCEWithLogitsLoss()
print("Params:", sum(p.numel() for p in model.parameters()) // 1_000_000, "M")


In [ ]:
# @markdown ## 7. Training loop (tracking val AUC, simpan bobot terbaik)
from sklearn.metrics import roc_auc_score

EPOCHS = 20
best_auc, best_state = 0.0, None
for ep in range(1, EPOCHS + 1):
    model.train()
    tot = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = crit(model(x).squeeze(1), y)
        loss.backward(); opt.step()
        tot += loss.item() * len(x)
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in val_dl:
            x = x.to(device)
            ys += y.tolist()
            ps += torch.sigmoid(model(x)).squeeze(1).cpu().tolist()
    auc = roc_auc_score(ys, ps)
    print(f"epoch {ep:2d} | loss {tot/len(tr):.4f} | val_auc {auc:.4f}", flush=True)
    if auc > best_auc:
        best_auc = auc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
print(f"\nBEST val_auc = {best_auc:.4f}")


In [ ]:
# @markdown ## 8. Evaluasi final (hold-out)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

model.load_state_dict(best_state)
model.eval()
ys, ps = [], []
with torch.no_grad():
    for x, y in val_dl:
        x = x.to(device)
        ys += y.tolist()
        ps += torch.sigmoid(model(x)).squeeze(1).cpu().tolist()
yp = (np.array(ps) >= 0.5).astype(int)
print("Accuracy :", round(accuracy_score(ys, yp), 4))
print("Precision:", round(precision_score(ys, yp), 4))
print("Recall   :", round(recall_score(ys, yp), 4))
print("F1       :", round(f1_score(ys, yp), 4))
print("AUC      :", round(roc_auc_score(ys, ps), 4))
print("Confusion ([[normal, tb] per baris]):\n", confusion_matrix(ys, yp))


In [ ]:
# @markdown ## 9. Simpan bobot → Google Drive (HANYA bobot yang keluar dari Colab)
from google.colab import drive
drive.mount("/content/drive")

import torch, os
save = {
    "state_dict": best_state,
    "pathologies": ["Normal", "Tuberculosis"],
    "val_auc": best_auc,
    "dataset": "Montgomery+Shenzhen (NLM/OpenI)",
    "note": "alat skrining/triase — bukan alat diagnostik; perlu validasi klinis",
}
out_dir = "/content/drive/MyDrive/orp-ai"
os.makedirs(out_dir, exist_ok=True)
path = os.path.join(out_dir, "tb_densenet121.pt")
torch.save(save, path)
print("Tersimpan:", path, f"({os.path.getsize(path)/1e6:.1f} MB)")
print("\nKemudian di lokal: taruh file ini di ai-worker/weights/tb_densenet121.pt")


## Langkah balik ke lokal (setelah training selesai)

1. Ambil `tb_densenet121.pt` dari `MyDrive/orp-ai/` (atau unduh langsung dari runtime).
2. Letakkan di: `ai-worker/weights/tb_densenet121.pt`.
3. `orp-ai infer <gambar>` otomatis melaporkan probabilitas TB bersama 18 patologi paru (integrasi modul `tb.py`).
4. Hasil ini **bukan diagnosis** — tetap alat bantu triase. Validasi klinis = tahap deployment.

> Catatan: runtime Colab ephemeral — dataset ±100 MB hilang saat sesi ditutup (itu memang tujuannya: tidak membebani penyimpanan lokal).
